In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime
from delta.tables import DeltaTable
import uuid

In [0]:
spark.sql("use catalog datatocurnch_novacart_adb")
spark.sql("create schema if not exists gold_schema")
gold_run_id=str(uuid.uuid4())

run_ts_str=datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

run_date_str=datetime.utcnow().strftime("%Y-%m-%d")
print("current gold run_id",gold_run_id)
print("run timestemp folder:",run_ts_str)

In [0]:
spark.sql("""
          
          create table if not exists datatocurnch_novacart_adb.gold_schema.processing_control(
              layer string,
              entity_name string,
              last_processed_silver_run_id string,
              last_processed_silver_run_ts timestamp,
              rows_merged bigint,
              run_status string,
              gold_run_id string,
              updated_at timestamp
          )
          using delta 
          """)

In [0]:
def upsert_to_gold(df_source,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        dt=DeltaTable.forName(spark,target_table)
        dt.alias("target").merge(
            df_source.alias("source"),
            "target."+join_key+" = source."+join_key
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_silver_ts(entity_name:str):
    ctrl=spark.table("datatocurnch_novacart_adb.gold_schema.processing_control")\
       .filter(
           (F.col("layer")=="gold") &
           (F.col("entity_name")==entity_name)&
           (F.col("run_status")=="success")
       ).orderBy(F.col("updated_at").desc()).limit(1)
    rows=ctrl.collect()
    if len(rows)==0:
       return None
    else:
       return rows[0].last_processed_silver_ts

In [0]:
def upsert_gold_control(entity_name,last_processed_silver_run_id,last_processed_run_ts,rows_merged):
    ctrl_df=spark.createDataFrame(
        [
            "gold",
            entity_name,
            last_processed_silver_run_id,
            last_processed_run_ts,
            rows_merged,
            "success",
            gold_run_id,
            run_ts_str,
            datetime.utcnow()
        ],
        schema="""
        layer string,
        entity_name string,
        last_processed_silver_run_id string,
        last_processed_silver_run_ts timestamp,
        rows_merged bigint,
        run_status string,
        gold_run_id string,
        updated_at timestamp
        """  
    )
    print(spark)
    dt=DeltaTable.forName(spark,"datatocurnch_novacart_adb.gold_schema.processing_control")
    dt.alias("target").merge(
        ctrl_df.alias("source"),
        "target.entity_name = source.entity_name"
    ).whenMatchedUpdate(set={
        "last_processed_silver_run_id":"s.last_processed_silver_run_id",
        "last_processed_silver_run_ts":"s.last_processed_silver_run_ts",
        "rows_merged":"s.rows_merged",
        "run_status":"s.run_status",
        "gold_run_id":"s.gold_run_id",
        "updated_at":"s.updated_at"
    }).whenNotMatchedInsertAll().execute();
     

In [0]:
last_gold_ts=get_last_processed_silver_ts("orders_information")
print("Last processed Silver Timestamp for GOLD:",last_gold_ts)

silver_orders_current=spark.read.table("datatocurnch_novacart_adb.silver_schema.orders_transformed")
silver_products_current=spark.read.table("datatocurnch_novacart_adb.silver_schema.products_transformed")
silver_payments_current=spark.read.table("datatocurnch_novacart_adb.silver_schema.payments_transformed")

if last_gold_ts is None:
    changed_orders=silver_orders_current
    changed_products=silver_products_current
    changed_payments=silver_payments_current
else:
    changed_orders=silver_orders_current.filter(F.col("updated_at")>F.lit(last_gold_ts))
    changed_products=silver_products_current.filter(F.col("updated_at")>F.lit(last_gold_ts))   
    changed_payments=silver_payments_current.filter(F.col("processed_at")>F.lit(last_gold_ts))   
    
print("Number of changed orders:",changed_orders.count())
print("Number of changed products:",changed_products.count())
print("Number of changed payments:",changed_payments.count())

In [0]:
impacted_from_orders=changed_orders.select("order_id").distinct()
impacted_from_products=(
    changed_products.alias("p")
    .join(silver_orders_current.alias("o"),
    F.col("p.product_id")==F.col("o.product_id"),"inner")
).select(F.col("o.order_id")).distinct()
impacted_from_payments=changed_payments.select("order_id").distinct()

impacted_order_ids=(
    impacted_from_orders
    .union(impacted_from_products)
    .union(impacted_from_payments)
    .distinct()
)

print("Impacted_order_ids:",impacted_order_ids.count())
display(impacted_order_ids.orderBy("order_id"))

In [0]:
impacted_orders = (
    silver_orders_current.alias("so")
    .join(
        impacted_order_ids.alias("i"),
        F.col("so.order_id") == F.col("i.order_id"),
        "inner"
    )
    .select("so.*")
)

gold_delta = (
    impacted_orders.alias("o")
    .join(
        silver_products_current.alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "inner"
    )
    .join(
        silver_payments_current.alias("pay"),
        F.col("o.order_id") == F.col("pay.order_id"),
        "inner"
    )
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("o.product_id"),
        F.col("o.order_status"),
        F.col("o.order_amount"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("pay.payment_id"),
        F.col("p.price").alias("product_price"),
        F.col("pay.payment_status"),
        F.col("pay.paid_amount"),
        F.col("o.order_date"),
        F.col("o.order_month"),
        F.col("o.order_year"),
        F.greatest(
            F.col("o.updated_at").cast("timestamp"),
            F.col("pay.processed_at").cast("timestamp"),
            F.col("p.updated_at").cast("timestamp")
        ).alias("gold_update_ts")
    )
    .dropDuplicates(["order_id"])
    .withColumn(
        "payment_completion_ratio",
        F.when(
            F.col("order_amount") > 0,
            F.col("paid_amount") / F.col("order_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "payment_state",
        F.when(F.col("payment_completion_ratio") == 1.0, "Paid")
         .when(F.col("payment_completion_ratio") == 0.0, "Unpaid")
         .when(F.col("payment_completion_ratio") < 1, "Partially_paid")
         .when(F.col("payment_completion_ratio") > 1, "Overpaid")
    )
    .withColumn(
        "gold_update_date",
        F.to_date(F.col("gold_update_ts"))
    )
    .withColumn(
        "gold_run_id",
        F.lit(gold_run_id)
    )
)

print("gold_delta_rows:", gold_delta.count())

display(gold_delta)

In [0]:
if gold_delta.count() > 0:
    upsert_to_gold(gold_delta,"datatocurnch_novacart_adb.gold_schema.orders_information","order_id")
else:
    print("No new rows to insert")

In [0]:
if not spark.catalog.tableExists("datatocurnch_novacart_adb.gold_schema.orders_information_scd2"):
    spark.sql("""
        CREATE TABLE datatocurnch_novacart_adb.gold_schema.orders_information_scd2
        USING DELTA
        AS
        SELECT *,
        cast(null as timestamp) as valid_from_ts,
        cast(null as timestamp) as valid_to_ts,
        true as is_current
        FROM datatocurnch_novacart_adb.gold_schema.orders_information
        WHERE 1 = 0
    """)

if gold_delta.count() > 0:
    gold_delta.createOrReplaceTempView("gold_delta_view")

    spark.sql("""
        MERGE INTO datatocurnch_novacart_adb.gold_schema.orders_information_scd2 AS t
        USING gold_delta_view AS s
        ON t.order_id = s.order_id AND t.is_current = true
        WHEN MATCHED AND (
            NOT (t.order_status <=> s.order_status) OR
            NOT (t.order_amount <=> s.order_amount) OR
            NOT (t.product_name <=> s.product_name) OR
            NOT (t.category <=> s.category) OR
            NOT (t.product_price <=> s.product_price) OR
            NOT (t.payment_id <=> s.payment_id) OR
            NOT (t.paid_amount <=> s.paid_amount) 
        )
        THEN UPDATE SET
            t.valid_to_ts = s.gold_update_ts,
            t.is_current = false
    """)

    spark.sql("""
        INSERT INTO datatocurnch_novacart_adb.gold_schema.orders_information_scd2
        SELECT s.*,
        s.gold_update_ts AS valid_from_ts,
        CAST(NULL AS timestamp) AS valid_to_ts,
        true AS is_current
        FROM gold_delta_view s
        LEFT JOIN datatocurnch_novacart_adb.gold_schema.orders_information_scd2 t
            ON s.order_id = t.order_id AND t.is_current = true
        WHERE t.order_id IS NULL OR (
            NOT (t.order_status <=> s.order_status) OR
            NOT (t.order_amount <=> s.order_amount) OR
            NOT (t.paid_amount <=> s.paid_amount) OR
            NOT (t.payment_id <=> s.payment_id) OR
            NOT (t.category <=> s.category) OR
            NOT (t.product_name <=> s.product_name) OR
            NOT (t.product_price <=> s.product_price)
        )
    """)

In [0]:
if gold_delta.count() > 0:
    impacted_categories = (
        gold_delta.select("category")
        .filter(F.col("category").isNotNull())
        .distinct()
    )

    category_performance_delta = (
        spark.read.table("datatocurnch_novacart_adb.gold_schema.orders_information")
        .join(impacted_categories, on="category", how="inner")
        .groupBy("category")
        .agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.sum(
                F.when(F.col("order_amount") > 0, F.col("order_amount")).otherwise(F.lit(0.0))
            ).alias("Gross_Merchandise_Value"),
            F.sum(
                F.when(F.col("paid_amount") > 0, F.col("paid_amount")).otherwise(F.lit(0.0))
            ).alias("Total_Paid_Amount"),
            F.avg(F.col("payment_completion_ratio")).alias("Average_Payment_Completion_Ratio"),
            (
                F.sum(
                    F.when(F.col("payment_status") == "FAILED", 1).otherwise(0)
                ) / F.count("*")
            ).alias("Payment_Failure_Rate")
        )
    )

    upsert_to_gold(
        category_performance_delta,
        "datatocurnch_novacart_adb.gold_schema.category_performance",
        "category"
    )
     

In [0]:
%sql
select * from datatocurnch_novacart_adb.gold_schema.category_performance

In [0]:
spark.sql("""
          create volume if not exists datatocurnch_novacart_adb.gold_schema.gold_snapshots_vol
          """)

In [0]:
latest_orders_path=(
    "/Volumes/datatocurnch_novacart_adb/gold_schema/gold_snapshots_vol/gold_latest/orders_information"
)
latest_category_path=(
    "/Volumes/datatocurnch_novacart_adb/gold_schema/gold_snapshots_vol/gold_latest/category_performance"
)
historical_orders_path=f"/Volumes/datatocurnch_novacart_adb/gold_schema/gold_snapshots_vol/gold_snapshots/orders_information/run_date={run_date_str}/run_ts={run_ts_str}"
historical_category_path=f"/Volumes/datatocurnch_novacart_adb/gold_schema/gold_snapshots_vol/gold_snapshots/category_performance/run_date={run_date_str}/run_ts={run_ts_str}"

In [0]:
spark.read.table("datatocurnch_novacart_adb.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(latest_orders_path)
spark.read.table("datatocurnch_novacart_adb.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(latest_category_path)

spark.read.table("datatocurnch_novacart_adb.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(historical_orders_path)
spark.read.table("datatocurnch_novacart_adb.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(historical_category_path)

In [0]:
print(f"latest_orders_path: {latest_orders_path}")
print(f"latest_category_path: {latest_category_path}")
print(f"historical_orders_path: {historical_orders_path}")
print(f"historical_category_path: {historical_category_path}")

In [0]:
latest_silver_ts=silver_orders_current.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]['mx']
latest_silver_run_id=(
    silver_orders_current
    .filter(F.col("bronze_ingested_at") == latest_silver_ts)
    .agg(F.max("silver_run_id").alias("mx"))
    .collect()[0]['mx']
)if latest_silver_ts is not None else None

gold_count = gold_delta.count()

print("latest_silver_ts =", latest_silver_ts)
print("latest_silver_run_id =", latest_silver_run_id)
print("gold_count =", gold_count)
print(spark)

ctrl_df = spark.createDataFrame(
    [(
        "gold",
        "orders_information",
        latest_silver_run_id,
        latest_silver_ts,
        gold_count,
        "success",
        gold_run_id,
        datetime.utcnow()
    )],
    schema="""
    layer string,
    entity_name string,
    last_processed_silver_run_id string,
    last_processed_silver_run_ts timestamp,
    rows_merged bigint,
    run_status string,
    gold_run_id string,
    updated_at timestamp
    """
)

dt = DeltaTable.forName(spark, "datatocurnch_novacart_adb.gold_schema.processing_control")
dt.alias("target").merge(
    ctrl_df.alias("source"),
    "target.entity_name = source.entity_name"
).whenMatchedUpdate(set={
    "last_processed_silver_run_id": "source.last_processed_silver_run_id",
    "last_processed_silver_run_ts": "source.last_processed_silver_run_ts",
    "rows_merged": "source.rows_merged",
    "run_status": "source.run_status",
    "gold_run_id": "source.gold_run_id",
    "updated_at": "source.updated_at"
}).whenNotMatchedInsertAll().execute()

display(spark.table("datatocurnch_novacart_adb.gold_schema.processing_control"))